# 06.3 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/03-choosing-k/exercises.ipynb)

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

In [ ]:
names = list("ABCDEFGH")
points = np.array([[1, 1], [1, 3], [2, 2], [4, 3], [6, 6], [7, 5], [7, 8], [9, 7]], dtype=float)
labels = np.array([0, 0, 0, 0, 1, 1, 1, 1])            # final clusters from 06.2

## Exercise 1: silhouette of E

By hand, compute the silhouette of **E (6, 6)** in the two-cluster solution. Then print `silhouette_samples` for all eight to check.

In [ ]:
# your code here

## Setup for exercises 2–3

In [ ]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

snapshot = sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)    # 2011-12-10
rfm = sales.groupby("Customer ID").agg(
    recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
    frequency=("Invoice", "nunique"),
    monetary=("Revenue", "sum"))
print(f"{len(sales):,} rows, {len(rfm):,} customers, snapshot {snapshot.date()}")

In [ ]:
from sklearn.preprocessing import StandardScaler

rfm_log = np.log1p(rfm)                         # tame the long right tails
X = StandardScaler().fit_transform(rfm_log)      # mean 0, std 1 per column
print("X shape:", X.shape)

In [ ]:
def profile(labels):
    """Size, mean R/F/M and revenue share per cluster, sorted by mean spend."""
    p = rfm.assign(cluster=labels).groupby("cluster").agg(
        customers=("recency", "size"), recency=("recency", "mean"),
        frequency=("frequency", "mean"), monetary=("monetary", "mean"))
    p["revenue_share"] = rfm.groupby(labels)["monetary"].sum() / rfm["monetary"].sum()
    return p.sort_values("monetary", ascending=False)

## Exercise 2: who is stuck between groups?

Fit K-Means with $k = 4$ (`n_init=10, random_state=42`). Compute `silhouette_samples` and draw a histogram of the silhouettes for each cluster. Which cluster has the largest share of negative values? Why might that be?

In [ ]:
# your code here

## Exercise 3: five groups, means and medians

Profile $k = 5$ with both means and medians of R, F and M. Which of the four personas splits? Would marketing treat the two halves differently?

In [ ]:
# your code here

## Exercise 4: the recommendation

In a markdown cell, write a short paragraph to the marketing director recommending a number of segments. Use at least one metric and at least one business argument.

*Your paragraph here.*